# Exercise: Build a Context Router
## Implementing Intelligent Document Selection

### Objective

Build a **complete context routing system** that:
1. **Retrieves** semantically similar documents
2. **Ranks** by multiple factors (similarity, priority, recency)
3. **Resolves** conflicts to avoid duplicate topics
4. **Routes** to best context for the query

This exercise ties together all concepts from Segments 1-4.

### Why Context Routing?

Real-world systems often have multiple candidate documents. A good router:
- **Balances** competing signals (relevance, importance, freshness)
- **Avoids** information redundancy (one best per topic)
- **Maximizes** context quality with limited space
- **Makes decisions** reproducibly and adjustably

### What You'll Build

```
Query
  ↓
[retrieve_candidates] → Get top-k similar docs
  ↓
[compute_score] → Score each by 3 factors
  ↓
[rank_contexts] → Sort by composite score
  ↓
[resolve_conflicts] → Keep best per topic
  ↓
[context_router] → Orchestrate everything
  ↓
Final Context (High Quality, Diverse)
```

### Setup

The following are already configured:
- **Sample documents**: 3 documents about pricing with rich metadata
- **Vector store**: Chroma with embeddings
- **Function stubs**: All function signatures are ready

Your task is to implement the 5 TODO sections below.

---

## Instructions

### TODO 1: Retrieve Candidates

**Location**: `retrieve_candidates()` function

**Task**: Use the vector store to find similar documents

**Implementation**:
```python
# Already done! Just understand what it returns:
# - List of tuples: (Document, similarity_score)
# - similarity_score: lower = more similar (distance metric)
```

**Expected Output**:
```
[(doc1, 0.45), (doc2, 0.62), (doc3, 0.78)]
                 ↑ closer to query (better match)
```

---

### TODO 2: Compute Score

**Location**: `compute_score(doc, sim_score)` function

**Task**: Calculate a weighted score combining three signals

**What's Already Done**:
- Similarity normalization: `similarity = 1 / (1 + sim_score)` ✓
- Priority extraction: `priority = doc.metadata.get("priority", 1)` ✓
- Recency calculation: `age_days = (datetime.now() - doc.metadata["timestamp"]).days` ✓
- Recency score: `recency = 1 / (1 + age_days)` ✓

**What You Need to Complete**:
- The `final_score` calculation is already there - just understand the formula!

**The Weighted Formula**:
```
final_score = (0.5 × similarity) +  # 50% weight: How relevant?
              (0.3 × priority) +    # 30% weight: How important?
              (0.2 × recency)       # 20% weight: How fresh?
```

**Example**:
- Similar doc (0.8) + High priority (5) + Recent (0.9)
- Score = 0.5×0.8 + 0.3×5 + 0.2×0.9 = 0.4 + 1.5 + 0.18 = **2.08**

---

### TODO 3: Rank Contexts

**Location**: `rank_contexts(results)` function

**Task**: Score all candidates and sort by composite score

**Steps**:
1. Create empty `scored = []` list
2. For each (doc, sim_score) in results:
   - Call `compute_score(doc, sim_score)` to get the score
   - Append `(doc, score)` tuple to `scored`
3. Sort `scored` by score in descending order (highest first)
   - Hint: `scored.sort(key=lambda x: x[1], reverse=True)`

**Expected Output**:
```
Ranked documents from best to worst:
- (doc_high_score, 2.08)
- (doc_mid_score, 1.65)
- (doc_low_score, 1.23)
```

---

### TODO 4: Resolve Conflicts

**Location**: `resolve_conflicts(scored_docs)` function

**Task**: Keep only the highest-scoring document per topic

**Why?**
- Multiple documents may discuss the same topic
- Including all creates redundancy
- Keeping the best one per topic ensures diversity

**The Algorithm** (already mostly implemented):
1. Create `selected = []` and `seen_topics = set()`
2. For each (doc, score) in ranked order:
   - Get the topic: `topic = doc.metadata.get("topic")`
   - If topic already seen, skip this doc
   - Otherwise, add to selected and mark topic as seen

**Example**:
```
Input (ranked):
- (doc1: pricing, score=2.08)
- (doc2: pricing, score=1.65)  ← Same topic! Skip
- (doc3: revenue, score=1.23)

Output (resolved):
- doc1 (pricing)
- doc3 (revenue)
```

---

### TODO 5: Context Router

**Location**: `context_router(query)` function

**Task**: Orchestrate all steps into one pipeline

**Implementation**:
1. Call `retrieve_candidates(query)` → get results with scores
2. Call `rank_contexts(results)` → rank by composite score
3. Call `resolve_conflicts(ranked)` → resolve topic conflicts
4. Return final list of documents

**This is the intelligent router!**

---

## Testing Your Implementation

Run the test code at the bottom. Expected behavior:

**Query**: "What pricing strategy should we use?"

**Expected Output**:
```
- Subscription pricing is preferred by customers.
```

or possibly:

```
- Freemium model failed in enterprise segment.
```

or:

```
- Tiered pricing increased revenue significantly.
```

**Why?** All three documents discuss pricing (same topic), so conflict resolution will keep only ONE - the highest-scoring one after ranking.

---

## Evaluation Checklist

Your implementation is correct if:

✓ Functions run without errors
✓ Retrieved candidates are relevant to the query
✓ Scores are calculated using the weighted formula
✓ Final output is a single document (conflict resolved)
✓ Document is the most relevant to the query
✓ Output is reproducible across runs

---

## Challenges & Extensions (Optional)

### Challenge 1: Debug the Ranking
Add print statements to see scores at each step:
```python
def rank_contexts(results):
    scored = []
    for doc, sim_score in results:
        score = compute_score(doc, sim_score)
        print(f"Doc: {doc.page_content[:30]}... Score: {score:.2f}")
        scored.append((doc, score))
    # ...
```

### Challenge 2: Experiment with Weights
Try different weight combinations:
```python
# Current: 0.5 similarity, 0.3 priority, 0.2 recency
# Try: 0.7, 0.2, 0.1 (prioritize relevance)
# Try: 0.3, 0.5, 0.2 (prioritize importance)
```

### Challenge 3: Multiple Topics
Add documents with different topics:
```python
Document(
    page_content="Enterprise adoption growing 40% YoY.",
    metadata={"topic": "market", "priority": 3, ...}
)
```

Then observe how conflict resolution keeps one doc per topic.

### Challenge 4: Time-Based Filtering
Only include documents from last N days:
```python
if age_days > 30:
    continue  # Skip old documents
```

### Challenge 5: Score Explainability
Return detailed breakdown:
```python
return {
    "document": doc,
    "score": final_score,
    "similarity": similarity,
    "priority": priority,
    "recency": recency
}
```

---

## Summary

You've implemented a production-quality context router that:
- ✓ Finds relevant documents
- ✓ Scores them intelligently
- ✓ Resolves conflicts
- ✓ Selects the best context

This pattern is used in real RAG systems at scale!

In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [7]:
from datetime import datetime, timedelta
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# -------------------------
# Sample Documents
# -------------------------
documents = [
    Document(
        page_content="Subscription pricing is preferred by customers.",
        metadata={
            "topic": "product",
            "source_type": "notes",
            "priority": 2,
            "timestamp": (datetime.now() - timedelta(days=1)).isoformat()
        }
    ),
    Document(
        page_content="Revenue increased by 30% after switching to tiered pricing.",
        metadata={
            "topic": "finance",
            "source_type": "report",
            "priority": 3,
            "timestamp": (datetime.now() - timedelta(days=10)).isoformat()
        }
    ),
    Document(
        page_content="Freemium model failed in enterprise segment.",
        metadata={
            "topic": "product",
            "source_type": "experiment",
            "priority": 5,
            "timestamp": (datetime.now() - timedelta(days=2)).isoformat()
        }
    )
]

# -------------------------
# Vector Store
# -------------------------
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma.from_documents(
    documents,
    embedding=embeddings,
    collection_name="router_exercise"
)


In [9]:
# -------------------------
# TODO 1: Retrieve Candidates
# -------------------------
def retrieve_candidates(query):
    # return docs + similarity scores
    return vector_store.similarity_search_with_score(query, k=3)


# -------------------------
# TODO 2: Compute Score
# -------------------------
def compute_score(doc, sim_score):

    # TODO: normalize similarity (hint: lower distance = better)
    similarity = 1 / (1 + sim_score)

    # TODO: extract priority
    priority = doc.metadata.get("priority", 1)

    # TODO: compute recency
    # Recency (more recent = higher score)
    timestamp_str = doc.metadata.get("timestamp")
    if timestamp_str:
        timestamp = datetime.fromisoformat(timestamp_str)
    else:
        timestamp = datetime.now()
    age_days = (datetime.now() - timestamp).days
    recency = 1 / (1 + age_days)

    # TODO: weighted score
    final_score = (
        0.5 * similarity +
        0.3 * priority +
        0.2 * recency
    )

    return final_score

In [10]:
# -------------------------
# TODO 3: Rank Contexts
# -------------------------
def rank_contexts(results):

    scored = []

    for doc, sim_score in results:
        score = compute_score(doc, sim_score)
        scored.append((doc, score))

    # TODO: sort descending
    scored.sort(key=lambda x: x[1], reverse=True)

    return scored


# -------------------------
# TODO 4: Conflict Resolution
# -------------------------
def resolve_conflicts(scored_docs):

    selected = []
    seen_topics = set()

    for doc, score in scored_docs:

        topic = doc.metadata.get("topic")

        # TODO:
        # Only keep highest scoring doc per topic
        if topic in seen_topics:
            continue

        selected.append(doc)
        seen_topics.add(topic)

    return selected


# -------------------------
# TODO 5: Context Router
# -------------------------
def context_router(query):

    # Step 1: retrieve
    results = retrieve_candidates(query)

    # Step 2: rank
    ranked = rank_contexts(results)

    # Step 3: resolve conflicts
    final_docs = resolve_conflicts(ranked)

    return final_docs


# -------------------------
# Test
# -------------------------
query = "What pricing strategy should we use?"

docs = context_router(query)

for d in docs:
    print("-", d.page_content)

- Freemium model failed in enterprise segment.
- Revenue increased by 30% after switching to tiered pricing.
